# Market Price Model — Marine Fishing AI (Kerala)

**⚠️ This trains on `data/mock/mock_monthly_prices.csv` — SYNTHETIC data, not real market history.**

Real price data currently covers exactly one week (24-30 Mar 2025), which isn't enough to show any seasonal
pattern. This notebook trains against a documented synthetic dataset (real anchor prices from that one week,
scaled by an assumed seasonal pattern tied to Kerala's known trawling-ban timing — see
`src/ml/generate_mock_price_data.py` for the exact assumption). Treat everything below as a **pipeline/prototype
validation**, not a real market-price predictor. When real multi-month price data exists, re-run this notebook
against that instead — the code doesn't need to change, only the input file.

**Model design notes (learned the hard way — read before changing the feature engineering or model type):**
1. **Two Fourier harmonics for month, not one.** A single `sin`/`cos` pair can only represent one symmetric
   wave (peak and trough exactly 6 months apart). This data's true pattern is a sharp 2-month peak (Jun-Jul,
   the trawling ban) with a dip only ~3-4 months later (Oct) — asymmetric, so it needs a second harmonic
   (`sin`/`cos` at double frequency) to represent properly. With only one harmonic, the fit forces a fake
   trough around December instead of the real one in October.
2. **Fit on `log(price)`, not raw price.** The mock data is generated *multiplicatively*
   (`price = anchor × seasonal_multiplier`), but a linear model with additive species dummy variables can only
   add or subtract the same flat Rupee amount to every species in a given month — it can't apply "the same
   percentage" to different baseline prices. Without the log transform, low-priced species (oil sardine) get
   wildly distorted while high-priced species (cuttlefish) barely move, because both get the same *absolute*
   seasonal swing. Taking `log(price)` turns the multiplication into addition, which an additive model can
   represent correctly — predictions are exponentiated back with `np.exp()` afterward.

Per the project's data-pipeline design: all data prep/cleaning logic lives in the repo (`src/ml/`), never
duplicated here. This notebook only does: load → train → evaluate → export.

## 1. Clone the repo and install dependencies

In [ ]:
# Replace with your actual GitHub repo URL / branch if different
!git clone -b feature/model-training https://github.com/vishaldharsan27-ux/Marine_AI.git repo
%cd repo
!pip install -q -r requirements.txt scikit-learn joblib

## 2. Load the mock price data

If `data/mock/mock_monthly_prices.csv` isn't in the cloned branch yet, regenerate it directly:
`!python -m src.ml.generate_mock_price_data`

In [ ]:
import pandas as pd

df = pd.read_csv("data/mock/mock_monthly_prices.csv")
assert df["is_synthetic"].all(), "Expected every row to be flagged synthetic - stop if this fails"
print(f"{len(df)} rows, {df['species_key'].nunique()} species, months {sorted(df['month'].unique())}")
df.head(12)

## 3. Feature engineering

Features: species (one-hot, not a raw integer label — a linear model would otherwise wrongly assume species
are ordered) + month encoded as **two** Fourier harmonics (see design notes above for why one isn't enough).
Target: `price_inr_per_kg` (fit in log-space in the next cell).

In [ ]:
import numpy as np

df["month_sin1"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos1"] = np.cos(2 * np.pi * df["month"] / 12)
df["month_sin2"] = np.sin(4 * np.pi * df["month"] / 12)
df["month_cos2"] = np.cos(4 * np.pi * df["month"] / 12)
species_dummies = pd.get_dummies(df["species_key"], prefix="sp")
df = pd.concat([df, species_dummies], axis=1)

FEATURES = ["month_sin1", "month_cos1", "month_sin2", "month_cos2"] + list(species_dummies.columns)
TARGET = "price_inr_per_kg"
df[FEATURES + [TARGET]].head()

## 4. Train / holdout split and fit (on log-price)

Holding out 3 months (Jun, Sep, Dec — one from each seasonal regime: ban spike, post-monsoon dip, baseline) so
the evaluation actually checks whether the model learned the shape of the season, not just memorized every row.

Model is Linear Regression, not RandomForest — with only 9 training points per species after the holdout split,
a tree ensemble can't reliably resolve which of two close months should be slightly higher (tried this first;
it scrambled the month ordering). A smooth linear model over the harmonic features fits this kind of small
periodic series far more reliably.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

HOLDOUT_MONTHS = [6, 9, 12]
train_df = df[~df["month"].isin(HOLDOUT_MONTHS)]
test_df = df[df["month"].isin(HOLDOUT_MONTHS)]

model = LinearRegression()
model.fit(train_df[FEATURES], np.log(train_df[TARGET]))

preds = np.exp(model.predict(test_df[FEATURES]))
mae = mean_absolute_error(test_df[TARGET], preds)
print(f"Holdout MAE (months {HOLDOUT_MONTHS}): Rs.{mae:.1f}/kg")
print(f"(for scale: holdout prices range Rs.{test_df[TARGET].min():.0f} to Rs.{test_df[TARGET].max():.0f}/kg)")

## 5. Inspect predictions across the full year, per species

Sanity check: does the model actually reproduce the ban-season spike / post-monsoon dip pattern, or did it just
learn a flat per-species average? Remember to exponentiate predictions back out of log-space.

In [ ]:
df["predicted_price"] = np.exp(model.predict(df[FEATURES]))

pivot = df.pivot_table(index="month_name", columns="species_key", values="predicted_price", sort=False)
pivot = pivot.reindex(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
pivot.round(1)

## 6. Retrain on all months and export the artifact

Saved locally in the Colab runtime. Download it or push it back to a branch (e.g. via a GitHub PAT stored as a
Colab secret) to get it into `src/ml/models/` in the actual repo — don't hand-copy weights out of the notebook.

In [ ]:
import joblib
from pathlib import Path

final_model = LinearRegression()
final_model.fit(df[FEATURES], np.log(df[TARGET]))

Path("src/ml/models").mkdir(parents=True, exist_ok=True)
joblib.dump({
    "model": final_model,
    "features": FEATURES,
    "target_transform": "log",
    "trained_on": "SYNTHETIC mock data - data/mock/mock_monthly_prices.csv",
}, "src/ml/models/market_price_model_MOCK.joblib")

print("Saved to src/ml/models/market_price_model_MOCK.joblib")
print("NOTE the _MOCK suffix is intentional - do not rename it away until this is retrained on real data.")
print("NOTE target_transform=log - predictions from this model need np.exp() applied, see cell 5 above.")